# 实践项目 04：XGBoost 脑疾病表格分类

我们将在 Kaggle Notebook 中整理受试者级表格，完成缺失值处理、变量编码、XGBoost 训练、概率阈值选择和变量贡献分析。

## 实践任务
1. 确认每行的样本单位并填写目标列
2. 识别纯标识符、数值变量和类别变量
3. 检查缺失比例与类别分布
4. 使用 Pipeline 完成填补、编码和模型训练
5. 补全轻量 XGBoost 参数
6. 在验证集上选择分类阈值
7. 在测试集上输出混淆矩阵、ROC、PR 和变量重要性

## 需要保存的结果
- `task4_data_summary.png`
- `task4_confusion.png`
- `task4_roc_pr.png`
- `task4_importance.png`
- `task4_result.json`


## 输出
- `task4_data_summary.png`
- `task4_confusion.png`
- `task4_roc_pr.png`
- `task4_importance.png`
- `task4_result.json`


In [ ]:
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,roc_curve,precision_recall_curve
from sklearn.inspection import permutation_importance

SEED=42; OUT=Path('/kaggle/working');OUT.mkdir(exist_ok=True)
CSV_PATH=None      # 可填写具体路径
TARGET_COLUMN=None # 可填写目标列名
csvs=list(Path('/kaggle/input').glob('**/*.csv'))
assert csvs,'没有找到 CSV 数据。'
path=Path(CSV_PATH) if CSV_PATH else csvs[0]
df=pd.read_csv(path)
print(path,df.shape,df.head())

## 任务 1：确定目标列与样本单位

填写 `TARGET_COLUMN`。删除纯标识符列时保留患者分组信息，确认每行代表一个受试者。


In [ ]:
# TODO 1
if TARGET_COLUMN is None:
    likely=[c for c in df.columns if c.lower() in {'target','label','class','status','diagnosis','stroke','parkinsons'}]
    TARGET_COLUMN=likely[0] if likely else None
assert TARGET_COLUMN in df.columns,'请在配置单元填写 TARGET_COLUMN。'
df=df[df[TARGET_COLUMN].notna()].copy()
target_encoder=LabelEncoder()
y=pd.Series(target_encoder.fit_transform(df[TARGET_COLUMN].astype(str)),index=df.index,name=TARGET_COLUMN)
assert len(target_encoder.classes_)==2,f'目标列需要是二分类，当前类别为 {list(target_encoder.classes_)}'
X=df.drop(columns=[TARGET_COLUMN])
# TODO：删除姓名、流水号等纯标识列
print('class mapping:',dict(enumerate(target_encoder.classes_)))
print(y.value_counts(),X.dtypes.value_counts())

## 任务 2：缺失与类型检查


In [ ]:
# TODO 2：输出缺失率最高的 10 列、数值列和类别列
missing=None
numeric_cols=None
categorical_cols=None
print(missing)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.5))
y.value_counts().plot(kind='bar',ax=ax[0],title='class distribution')
(df.isna().mean().sort_values(ascending=False).head(12)).plot(kind='bar',ax=ax[1],title='missing proportion')
plt.tight_layout();plt.savefig(OUT/'task4_data_summary.png',dpi=160);plt.show()

## 3. 独立划分与预处理


In [ ]:
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=.30,stratify=y,random_state=SEED)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=.50,stratify=y_temp,random_state=SEED)
num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median'))])
cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
pre=ColumnTransformer([('num',num_pipe,numeric_cols),('cat',cat_pipe,categorical_cols)])

## 任务 3：建立 XGBoost 模型

优先使用 `xgboost.XGBClassifier`。若环境中没有 xgboost，使用 `HistGradientBoostingClassifier` 并只保留数值变量。


In [ ]:
try:
    from xgboost import XGBClassifier
    # TODO 3：填写一个轻量参数组合
    clf=XGBClassifier(n_estimators=None,max_depth=None,learning_rate=None,subsample=.8,colsample_bytree=.8,eval_metric='logloss',random_state=SEED)
    model=Pipeline([('pre',pre),('clf',clf)])
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    numeric_cols=list(X.select_dtypes(include=np.number).columns); categorical_cols=[]
    pre=ColumnTransformer([('num',num_pipe,numeric_cols)])
    model=Pipeline([('pre',pre),('clf',HistGradientBoostingClassifier(max_iter=150,max_depth=3,learning_rate=.05,random_state=SEED))])
model.fit(X_train,y_train)
val_prob=model.predict_proba(X_val)[:,1]
print('validation AUC',roc_auc_score(y_val,val_prob))

## 任务 4：阈值选择与测试评价


In [ ]:
# TODO 4：在验证集的候选阈值中选择 F1 最高者
thresholds=np.linspace(.2,.8,13)
best_threshold=None

test_prob=model.predict_proba(X_test)[:,1]
test_pred=(test_prob>=best_threshold).astype(int)
metrics={'accuracy':accuracy_score(y_test,test_pred),'f1':f1_score(y_test,test_pred),'roc_auc':roc_auc_score(y_test,test_prob),'pr_auc':average_precision_score(y_test,test_prob)}
cm=confusion_matrix(y_test,test_pred); print(metrics,cm)

In [ ]:
plt.figure(figsize=(4,4));plt.imshow(cm,cmap='viridis');plt.colorbar();plt.xlabel('predicted');plt.ylabel('true');plt.xticks([0,1]);plt.yticks([0,1]);plt.tight_layout();plt.savefig(OUT/'task4_confusion.png',dpi=160);plt.show()
fpr,tpr,_=roc_curve(y_test,test_prob);p,r,_=precision_recall_curve(y_test,test_prob)
fig,ax=plt.subplots(1,2,figsize=(8,3.5));ax[0].plot(fpr,tpr);ax[0].plot([0,1],[0,1],'--');ax[0].set_title('ROC');ax[1].plot(r,p);ax[1].set_title('Precision–Recall');plt.tight_layout();plt.savefig(OUT/'task4_roc_pr.png',dpi=160);plt.show()

## 任务 5：特征重要性与单变量对照


In [ ]:
# TODO 5：使用 permutation_importance 计算测试集前 12 个重要变量
pi=permutation_importance(model,X_test,y_test,n_repeats=5,random_state=SEED,scoring='roc_auc')
order=np.argsort(pi.importances_mean)[-12:]
plt.figure(figsize=(7,4));plt.barh(np.array(X.columns)[order],pi.importances_mean[order]);plt.xlabel('AUC decrease');plt.tight_layout();plt.savefig(OUT/'task4_importance.png',dpi=160);plt.show()

# 只修改 max_depth 或 learning_rate 之一，重新训练并记录验证 AUC
comparison={'changed_variable':None,'baseline_value':None,'new_value':None,'baseline_validation_auc':float(roc_auc_score(y_val,val_prob)),'new_validation_auc':None,'observation':None}
result={**metrics,'best_threshold':float(best_threshold),'target_classes':target_encoder.classes_.tolist(),'train_samples':len(X_train),'test_samples':len(X_test),'comparison':comparison,'seed':SEED}
(OUT/'task4_result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8');result

## 结论要求

用 300–500 字说明样本单位、缺失处理、测试表现、阈值影响以及最重要变量在模型中的作用。
